In [143]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import ast

In [157]:
df = pd.read_csv('movies.csv') 
df = df[['title', 'budget', 'revenue', 'genres', 'belongs_to_collection', 
          'release_date', 'vote_average', 'vote_count', 'runtime']]
df = df[(df['budget']>0) & (df['revenue']>0)] #remove any rows where either budget are 0/unknown 


In [158]:
#clean up genres, extract just name

def extract_genres(genres_str):
    genres = ast.literal_eval(genres_str) #convert to dictionary
    return [g['name'] for g in genres]

df['genres'] = df['genres'].apply(extract_genres)


In [190]:
#belongs_to_collection true/false 
df['belongs_to_collection'] = df['belongs_to_collection'].notna()
df = df.rename(columns={'belongs_to_collection': 'is_franchise'})

In [191]:
#return on investment for each movie
df['roi'] = (df['revenue'] - df['budget'])/df['budget']
df.head(20)

,title,budget,revenue,genres,is_franchise,release_date,vote_average,vote_count,runtime,roi
0,Karate Kid: Legends,45000000.0,104560790,"[Action, Adventure, Drama]",True,2025-05-08,7.281,367,94,1.323573
1,Ballerina,90000000.0,131611905,"[Action, Thriller, Crime]",True,2025-06-04,7.453,944,125,0.462355
2,Superman,225000000.0,217000000,"[Science Fiction, Adventure, Action]",True,2025-07-09,7.470,538,130,-0.035556
7,Jurassic World Rebirth,180000000.0,529463000,"[Science Fiction, Adventure, Action]",True,2025-07-01,6.400,566,134,1.941461
8,Thunderbolts*,180000000.0,382027956,"[Action, Science Fiction, Adventure]",True,2025-04-30,7.428,1767,127,1.122378
10,Final Destination Bloodlines,50000000.0,285153000,"[Horror, Mystery]",True,2025-05-14,7.200,1605,110,4.703060
12,Lilo & Stitch,100000000.0,994264677,"[Family, Science Fiction, Comedy, Adventure]",True,2025-05-17,7.154,827,108,8.942647
14,How to Train Your Dragon,150000000.0,560773000,"[Fantasy, Family, Action]",True,2025-06-06,7.887,630,125,2.738487
15,Bring Her Back,15000000.0,22878745,[Horror],True,2025-05-28,7.425,242,104,0.525250
16,F1,200000000.0,393395000,"[Action, Drama]",True,2025-06-25,7.664,727,156,0.966975


In [192]:
#remove duplicates, many are remakes so drop exact duplicates only (same release date)
df['title'].duplicated().sum() #172 duplicates
df = df.drop_duplicates(subset=['title', 'release_date']) 

In [193]:
df_genres = df.explode('genres')
df_genres.head(10)

,title,budget,revenue,genres,is_franchise,release_date,vote_average,vote_count,runtime,roi
0,Karate Kid: Legends,45000000.0,104560790,Action,True,2025-05-08,7.281,367,94,1.323573
0,Karate Kid: Legends,45000000.0,104560790,Adventure,True,2025-05-08,7.281,367,94,1.323573
0,Karate Kid: Legends,45000000.0,104560790,Drama,True,2025-05-08,7.281,367,94,1.323573
1,Ballerina,90000000.0,131611905,Action,True,2025-06-04,7.453,944,125,0.462355
1,Ballerina,90000000.0,131611905,Thriller,True,2025-06-04,7.453,944,125,0.462355
1,Ballerina,90000000.0,131611905,Crime,True,2025-06-04,7.453,944,125,0.462355
2,Superman,225000000.0,217000000,Science Fiction,True,2025-07-09,7.470,538,130,-0.035556
2,Superman,225000000.0,217000000,Adventure,True,2025-07-09,7.470,538,130,-0.035556
2,Superman,225000000.0,217000000,Action,True,2025-07-09,7.470,538,130,-0.035556
7,Jurassic World Rebirth,180000000.0,529463000,Science Fiction,True,2025-07-01,6.400,566,134,1.941461


In [195]:
engine = create_engine('postgresql://localhost/movies_db')
df.to_sql('movies', con=engine, if_exists='replace', index=False)
df_genres.to_sql(name='movie_genres', con=engine, if_exists='replace', index=False)

662